# 05 · Playstyle clusters (player pages)
**Question:** what kind of player is this? Cluster accounts on their last 50 games into a handful of archetypes, name each one, and show the label on `/player/:id`.
**Inputs:** `fe.PROFILE_COLS`: kills, deaths, assists, souls, last hits and denies per minute, KDA, win rate, hero pool entropy, top hero share.
Needs `python -m broadway_ml.pull histories --players 3000` first.
**No ground truth**, so judge by: silhouette score, stability across seeds and across two halves of the data, and whether the clusters can be named in a sentence.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
hist = pd.read_parquet('../data/tables/histories.parquet')
prof = fe.player_profiles(hist, last_n=50, min_matches=15)
prof.describe().T

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
F = fe.PROFILE_COLS  # TODO(V): win_rate may be better left out, it is outcome not style
Z = StandardScaler().fit(prof[F])
Xs = Z.transform(prof[F])
scores = {k: silhouette_score(Xs, KMeans(k, n_init=10, random_state=0).fit_predict(Xs)) for k in range(3, 10)}
pd.Series(scores).plot(marker='o', title='silhouette by k');

In [ ]:
# TODO(V): pick k, check stability, try GaussianMixture, add hero-class shares (spirit vs gun heroes) as features
K = 5
km = KMeans(K, n_init=20, random_state=0).fit(Xs)
centres = pd.DataFrame(km.cluster_centers_, columns=F).round(2)
centres['players'] = np.bincount(km.labels_)
centres  # read these in z-scores: +1 = one standard deviation above the average player

In [ ]:
LABELS = ['TODO'] * K  # name each cluster in the site's voice, e.g. 'Soul hoarder', 'Brawler', 'Support'
ex.export_centroids(km, F, LABELS, '../models/playstyle.json', scaler=Z, name='playstyle',
                    metrics={'k': K, 'silhouette': float(silhouette_score(Xs, km.labels_)), 'players': len(prof)})